In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
pip install fastapi uvicorn python-multipart scikit-learn joblib 

Note: you may need to restart the kernel to use updated packages.


Task 1.1: Load and Prepare Data 

In [2]:
import os
import pytesseract
from PIL import Image

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

import joblib


def load_documents(data_dir):
    """
    Load documents from folder structure
    and extract text using OCR.
    """

    documents = []
    labels = []

    # Loop through each folder
    for doc_type in os.listdir(data_dir):

        folder_path = os.path.join(data_dir, doc_type)

        # Skip non-folder files
        if not os.path.isdir(folder_path):
            continue

        # Loop through images inside folder
        for filename in os.listdir(folder_path):

            file_path = os.path.join(folder_path, filename)

            try:
                # Open image
                img = Image.open(file_path)

                # OCR text extraction
                text = pytesseract.image_to_string(img)

                # Store extracted text and label
                documents.append(text)
                labels.append(doc_type)

            except Exception as e:
                print(f"Error processing {file_path}: {e}")

    return documents, labels


# ==============================
# LOAD DATASET
# ==============================

documents, labels = load_documents("/kaggle/input/datasets/anderianisar/week-8")

print(f"Loaded {len(documents)} documents")
print(f"Classes: {set(labels)}")


# ==============================
# TEXT VECTORIZATION
# ==============================

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(documents)
y = labels


# ==============================
# TRAIN / TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ==============================
# TRAIN MODEL
# ==============================

model = LogisticRegression()

model.fit(X_train, y_train)


# ==============================
# EVALUATION
# ==============================

y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


# ==============================
# SAVE MODEL
# ==============================

joblib.dump(model, "document_classifier.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("\nModel and vectorizer saved successfully!")

Loaded 30 documents
Classes: {'contracts', 'receipts', 'invoices'}

Accuracy: 0.6666666666666666

Classification Report:

              precision    recall  f1-score   support

   contracts       0.50      0.50      0.50         2
    invoices       0.50      0.50      0.50         2
    receipts       1.00      1.00      1.00         2

    accuracy                           0.67         6
   macro avg       0.67      0.67      0.67         6
weighted avg       0.67      0.67      0.67         6


Model and vectorizer saved successfully!


Task 1.2: Train Classifier

In [4]:
# ==============================
# SPLIT DATA
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    documents,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)


# ==============================
# CREATE TF-IDF VECTORIZER
# ==============================

vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1, 2)   # Unigrams + Bigrams
)


# ==============================
# TRANSFORM TEXT DATA
# ==============================

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


# ==============================
# TRAIN CLASSIFIER
# ==============================

classifier = LogisticRegression(
    max_iter=1000
)

classifier.fit(X_train_vec, y_train)


# ==============================
# MODEL EVALUATION
# ==============================

y_pred = classifier.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))

Accuracy: 83.33%

Classification Report:

              precision    recall  f1-score   support

   contracts       0.67      1.00      0.80         2
    invoices       1.00      1.00      1.00         2
    receipts       1.00      0.50      0.67         2

    accuracy                           0.83         6
   macro avg       0.89      0.83      0.82         6
weighted avg       0.89      0.83      0.82         6



Task 1.3: Save Model 

In [7]:

import os
import joblib

# ==============================
# CREATE MODELS DIRECTORY
# ==============================

os.makedirs("models", exist_ok=True)


# ==============================
# SAVE VECTORIZER AND CLASSIFIER
# ==============================

joblib.dump(vectorizer, "models/vectorizer.pkl")
joblib.dump(classifier, "models/classifier.pkl")

print("Models saved successfully!")


# ==============================
# LOAD SAVED MODELS
# ==============================

loaded_vec = joblib.load("models/vectorizer.pkl")
loaded_clf = joblib.load("models/classifier.pkl")

print("Models loaded successfully!")

Models saved successfully!
Models loaded successfully!
